## 0. Setup

---
## 3.5 Compare ML Frameworks

We  train the **regression task** — predict `tip_amount` from trip
features — using:

| Framework | Library | Execution |
|-----------|---------|-----------|
| **scikit-learn** | `sklearn.linear_model.LinearRegression` | single-machine, pandas |
| **XGBoost** | `xgboost.XGBRegressor` | single-machine, pandas |
| **Spark ML** | `pyspark.ml.regression.LinearRegression` | distributed, Spark DataFrame |
| **XGBoost (distributed)** | `xgboost.spark.SparkXGBRegressor` | distributed, Spark DataFrame |

The data format (Parquet) and feature set stay constant — only the framework changes.

In [ ]:
import pandas as pd
# Load the Parquet data back into pandas for the single-machine frameworks
PARQUET_PATH = "/home/alka/MLOPs-Class/ML_Framework/taxi_tip_regression.parquet"
pdf = pd.read_parquet(PARQUET_PATH)
print(f"Pandas DataFrame: {pdf.shape[0]:,} rows x {pdf.shape[1]} columns")
pdf.head()

### 3.5.1 scikit-learn — `LinearRegression`

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler as SkStandardScaler
from sklearn.linear_model import LinearRegression as SkLinearRegression
from sklearn.metrics import mean_squared_error, r2_score
reg_features = ["trip_distance", "fare_amount", "passenger_count",
                "duration_min", "hour_of_day", "day_of_week"]
label_col    = "tip_amount"
X = pdf[reg_features].values
y = pdf[label_col].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

sk_scaler = SkStandardScaler().fit(X_train)
X_train_s = sk_scaler.transform(X_train)
X_test_s  = sk_scaler.transform(X_test)

sk_lr = SkLinearRegression().fit(X_train_s, y_train)
sk_preds = sk_lr.predict(X_test_s)

sk_rmse = mean_squared_error(y_test, sk_preds) ** 0.5
sk_r2   = r2_score(y_test, sk_preds)

print(f"scikit-learn  RMSE: {sk_rmse:.4f}  |  R²: {sk_r2:.4f}")

### 3.5.2 XGBoost — `XGBRegressor` (single machine)

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=100, max_depth=5, learning_rate=0.1,
    objective="reg:squarederror", random_state=42,
)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

xgb_rmse = mean_squared_error(y_test, xgb_preds) ** 0.5
xgb_r2   = r2_score(y_test, xgb_preds)

print(f"XGBoost       RMSE: {xgb_rmse:.4f}  |  R²: {xgb_r2:.4f}")

### 3.5.3 Spark ML — `LinearRegression` (distributed)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
# ML: features
from pyspark.ml.feature import VectorAssembler, StandardScaler

# ML: algorithms
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import (
    RegressionEvaluator,
    BinaryClassificationEvaluator,
    ClusteringEvaluator,
    MulticlassClassificationEvaluator,
)
spark = (
    SparkSession.builder
    .appName("SparkML_Taxi_Demo")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
# Read the same Parquet file back with Spark (distributed pipeline)
# Note: the Parquet file already contains the scaled features (from df_out),
# so we only need VectorAssembler — no StandardScaler here.
df_parquet = spark.read.parquet(PARQUET_PATH)
train_pq, test_pq = df_parquet.randomSplit([0.8, 0.2], seed=42)

sparkml_assembler = VectorAssembler(inputCols=reg_features, outputCol="features")
sparkml_lr        = LinearRegression(featuresCol="features", labelCol=label_col,
                                      maxIter=20, regParam=0.1)

sparkml_pipeline = Pipeline(stages=[sparkml_assembler, sparkml_lr])
sparkml_model    = sparkml_pipeline.fit(train_pq)
sparkml_preds    = sparkml_model.transform(test_pq)

sparkml_rmse = RegressionEvaluator(labelCol=label_col, metricName="rmse").evaluate(sparkml_preds)
sparkml_r2   = RegressionEvaluator(labelCol=label_col, metricName="r2").evaluate(sparkml_preds)

print(f"Spark ML      RMSE: {sparkml_rmse:.4f}  |  R²: {sparkml_r2:.4f}")


### 3.5.4 XGBoost Distributed — `SparkXGBRegressor`

In [ ]:
from xgboost.spark import SparkXGBRegressor

xgb_spark_assembler = VectorAssembler(inputCols=reg_features, outputCol="features")
train_xgb = xgb_spark_assembler.transform(train_pq)
test_xgb  = xgb_spark_assembler.transform(test_pq)

xgb_spark = SparkXGBRegressor(
    features_col="features", label_col=label_col,
    num_workers=2, n_estimators=100, max_depth=5, learning_rate=0.1,
)

xgb_spark_model = xgb_spark.fit(train_xgb)
xgb_spark_preds = xgb_spark_model.transform(test_xgb)

xgbspark_rmse = RegressionEvaluator(labelCol=label_col, metricName="rmse").evaluate(xgb_spark_preds)
xgbspark_r2   = RegressionEvaluator(labelCol=label_col, metricName="r2").evaluate(xgb_spark_preds)

print(f"XGBoost-Spark RMSE: {xgbspark_rmse:.4f}  |  R²: {xgbspark_r2:.4f}")